# 📊 Méthodologie Complète : Des Données Brutes au Dataset Final

## 🗂️ 1. POINT DE DÉPART : Dossier D4D Brut

### Contenu Initial
**Localisation :** `C:\Users\ALPHEE COLOMBE\Desktop\D4D_mobile-master\D4D_mobile-master\dat`
```
dat/
├── Abidjan.Commune.CallNetwork.txt
├── cities_departmentID.csv (1,231 antennes)
├── cities_regionID.csv
├── region_stats.txt
├── pole_pop.txt, pole_tower.txt
└── 15+ autres fichiers TXT
```

**Problème :** Données désorganisées, 20+ fichiers séparés

---

## 🔍 2. DÉCOUVERTE ET ANALYSE

### Fichier Consolidé Trouvé
**`D4D_MOBILITE_CI_FINAL.xlsx`** (dans dossier parent)

**8 feuilles :**
1. `flux_departements` (2,450 lignes) → **DONNÉE CLÉ**
2. `flux_sous_prefectures` (20,976 lignes)
3. `referentiel_geo` (255 sous-préfectures)
4. `mapping_antennes` (1,231 antennes)
5. `stats_globales` (résumé dataset)
6. `top_departements` (top 10)
7. `top_corridors` (top 10)
8. `metadonnees` (documentation)

### Comment ce Fichier a été Créé (Reconstruction Logique)

**ÉTAPE A - Géolocalisation :**
- Fusion `cities_departmentID.csv` + `cities_regionID.csv`
- Résultat : 1,231 antennes avec coordonnées GPS + département + région
- → Feuille `mapping_antennes`

**ÉTAPE B - Comptage Déplacements :**
- Logs bruts Orange : Handovers utilisateurs entre antennes
- Si Antenne A (département X) → Antenne B (département Y)
- Comptage : Nombre déplacements X → Y
- → Table FLUX_BRUTS

**ÉTAPE C - Enrichissement GPS :**
- Ajout coordonnées GPS centrales par département
- Calcul distances (formule Haversine)
- Calcul flux journalier = Total / 150 jours
- → Feuille `flux_departements`

**ÉTAPE D - Statistiques :**
- Agrégations : Sommes, moyennes, tops
- → Feuilles `stats_globales`, `top_departements`, `top_corridors`

**Résultat :** Fichier Excel structuré remplaçant 20+ fichiers bruts

---

## ⚠️ 3. PROBLÈME IDENTIFIÉ

### Écart Temporel Critique
- **Données disponibles :** Décembre 2011 - Avril 2012
- **Objectif projet :** Prédire COVID-19 (2020)
- **Écart :** 8-9 ans !

### Solution Adoptée
**Hypothèse :** Structure spatiale stable + Ajustement volume

---

## 🔧 4. COLLECTE DONNÉES COMPLÉMENTAIRES

### A. Google COVID-19 Mobility Reports

**Objectif :** Variations mobilité pendant COVID

**Téléchargement :**
- URL : Global_Mobility_Report.csv (811 Mo)
- Filtrage : "Côte d'Ivoire" (avec accent Ô)
- Résultat : 11,705 lignes quotidiennes (2020-2022)

**Contenu :** Variations % par catégorie (commerces, transports, travail)

**Exemple :**
- 15/04/2020 : Transports -70% (confinement)

### B. Our World in Data - COVID

**Objectif :** Cas réels pour validation

**Téléchargement :**
- URL : owid-covid-data.csv
- Filtrage : iso_code = 'CIV'
- Résultat : 1,674 jours (2020-2024)

**Contenu :** Date, nouveaux cas, total cas, décès, population

**Info clé :** Population 2020 = 28,160,548

### C. Banque Mondiale - Population

**Objectif :** Croissance démographique officielle

**API Banque Mondiale :**
- Indicateur : SP.POP.TOTL
- Période : 2011-2022

**Résultat officiel :**
- 2012 : 23,467,078 habitants
- 2020 : 28,915,449 habitants
- **Facteur : +23.2%** (1.2322)

---

## 🧹 5. NETTOYAGE DONNÉES

### Google Mobility
**Avant :** 811 Mo, 15+ colonnes  
**Actions :**
- Filtrage Côte d'Ivoire uniquement
- Sélection 9 colonnes utiles
- Renommage clair (commerces_pct, transports_pct...)

**Après :** 11,705 lignes, 9 colonnes

### Calcul Facteurs Mensuels
**Objectif :** Transformer variations quotidiennes en coefficients mensuels

**Méthode :**
- Grouper par mois
- Moyenne `transports_pct` par mois
- Convertir en multiplicateur

**Exemple avril 2020 :**
```
Moyenne mensuelle : -33%
Facteur = (1 + (-33/100)) = 0.67
```

**Résultat :** 34 facteurs mensuels (février 2020 - octobre 2022)

### COVID-19
**Avant :** 100+ colonnes  
**Actions :** Sélection 7 colonnes essentielles  
**Après :** 1,674 lignes, 7 colonnes

---

## 🔄 6. AJUSTEMENT FLUX 2012 → 2020

### Principe Double Ajustement

**Formule :**
```
Flux Final = Flux 2012 × Facteur Démographique × Facteur Mobilité Période
```

### Ajustement 1 : Démographique (+23.2%)

**Application :** Tous les flux × 1.2322

**Exemple Abidjan → Soubré :**
```
Flux 2012        : 35,872 pers/jour
Flux 2020 base   : 35,872 × 1.2322 = 44,203 pers/jour
```

**Justification :** Population +23.2% → Flux +23.2% (proportionnel)

### Ajustement 2 : Temporel COVID (Variable)

**Application :** Flux 2020 base × Facteur mois

**Avril 2020 (Confinement) :**
```
Flux 2020 base   : 44,203
Facteur avril    : 0.67 (confinement -33%)
Flux avril 2020  : 44,203 × 0.67 = 29,616 pers/jour
```

**Juillet 2021 (Reprise) :**
```
Flux 2020 base   : 44,203
Facteur juillet  : 1.43 (reprise +43%)
Flux juillet 2021: 44,203 × 1.43 = 63,210 pers/jour
```

**Interprétation :**
- Avril 2020 : -17% vs 2012 (confinement compense croissance)
- Juillet 2021 : +76% vs 2012 (croissance + reprise forte)

---

## 📦 7. CRÉATION DATASET FINAL

### Fichier : `DONNEES_FINALES_HACKATHON.xlsx`

**5 Feuilles :**

#### 1. `flux_ajuste_2020`
**Source :** flux_departements + ajustement démographique

**Colonnes ajoutées :**
- `flux_journalier_ajuste_2020` : Flux × 1.2322
- `weight_ajuste_2020` : Weight × 1.2322

**Usage :** Matrice mobilité 2020 de base

#### 2. `google_mobility`
**Source :** Google Mobility nettoyé

**Contenu :** 11,705 lignes quotidiennes, 9 colonnes

**Usage :** Variations quotidiennes (analyses fines)

#### 3. `covid_ci`
**Source :** Our World in Data nettoyé

**Contenu :** 1,674 jours, 7 colonnes

**Usage :** Validation modèle

#### 4. `facteurs_ajustement`
**Source :** Calcul depuis Google Mobility

**Contenu :** 34 mois, 2 colonnes (mois, facteur)

**Usage :** Application directe aux flux

#### 5. `population_officielle`
**Source :** Banque Mondiale

**Contenu :** 2011-2022, population annuelle

**Usage :** Documentation ajustement

---

## ✅ 8. VALIDATION

### Vérifications
- ✅ Cohérence spatiale : 50 départements, 2,450 corridors
- ✅ Cohérence temporelle : 2020-2022 couverts
- ✅ Cohérence démographique : Facteur 1.2322 validé

### Statistiques Finales
- Flux journalier total 2020 : ~1,97M déplacements/jour
- 5 feuilles consolidées
- 4 sources croisées (Orange, Google, OWID, Banque Mondiale)

---

## 🎯 9. RÉCAPITULATIF COMPLET

### Flux de Travail
```
ÉTAPE 1 : Dossier D4D brut (20+ fichiers TXT)
    ↓
ÉTAPE 2 : Découverte Excel consolidé (8 feuilles)
    ↓
ÉTAPE 3 : Identification problème temporel (2012 vs 2020)
    ↓
ÉTAPE 4 : Collecte données 2020
    • Google Mobility → Variations COVID
    • OWID → Cas réels
    • Banque Mondiale → Population officielle
    ↓
ÉTAPE 5 : Nettoyage
    • Filtrage CI uniquement
    • Suppression colonnes inutiles
    • Calcul facteurs mensuels
    ↓
ÉTAPE 6 : Ajustement flux 2012 → 2020
    • Démographique : × 1.2322
    • Temporel : × facteur mois
    ↓
ÉTAPE 7 : Consolidation finale
    • 5 feuilles Excel
    • Données 2020 ajustées
    ↓
✅ DONNEES_FINALES_HACKATHON.xlsx
```

### Transformations Clés

| Donnée | Avant | Après |
|--------|-------|-------|
| Flux | 2012 brut | 2020 ajusté (+23.2%) |
| Google | 811 Mo, 15 col | 11,705 lignes, 9 col |
| COVID | 100+ col | 7 col essentielles |
| Facteurs | % quotidiens | 34 multiplicateurs mensuels |

---

## 📊 10. UTILISATION PRATIQUE

### Pour Simulation Avril 2020

**Étape 1 :** Charger `flux_ajuste_2020` (base 2020)

**Étape 2 :** Appliquer facteur 0.67 (depuis `facteurs_ajustement`)
```
Flux avril = flux_ajuste_2020 × 0.67
```

**Étape 3 :** Initialiser cas COVID (depuis `covid_ci`)

**Étape 4 :** Lancer modèle SEIR avec flux avril

**Étape 5 :** Comparer prédictions vs `covid_ci` réel

---

## 🔍 11. LIMITATIONS

**Hypothèses assumées :**
1. Structure mobilité 2012 stable jusqu'en 2020
2. Ajustement démographique uniforme (+23.2% partout)
3. Données COVID nationales (pas de détail départemental)
4. Google Mobility agrégée (pas par département CI)

**Justifications :**
- Pas de données plus récentes disponibles
- Ajustements multiples compensent écart temporel
- Méthodologie acceptée en modélisation épidémiologique

---

## 📚 12. SOURCES

1. Orange D4D Challenge 2012
2. Google COVID-19 Mobility Reports
3. Our World in Data - COVID-19
4. Banque Mondiale - Population (API)

---

## ✅ CONCLUSION

**Dataset Final :**
- ✅ Flux 2012 ajustés → 2020 (+23.2%)
- ✅ Variations COVID 2020-2022 intégrées
- ✅ Cas réels pour validation
- ✅ Sources officielles documentées
- ✅ 5 feuilles consolidées prêtes à l'emploi

**PRÊT POUR :** Modélisation SEIR multi-compartiments avec validation COVID-19 réelle

**PROCHAINE ÉTAPE :** Construction du modèle épidémiologique